# 03 — Baseline Forecasts

The bar the ML models must clear. These are not straw men: seasonal naive exploits the strong weekly cycle, and Croston/SBA are the standard methods for the sparse series that dominate this panel.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
from src.data.loader import load_panel
cfg = load_config()
panel = load_panel(cfg)
print(f'panel: {len(panel):,} rows, {panel.series_id.nunique()} series, {panel.date.nunique()} days')

In [ ]:
from src.forecasting.baselines import BASELINE_DESCRIPTIONS
import pandas as pd
pd.DataFrame(BASELINE_DESCRIPTIONS.items(), columns=['baseline','definition'])

## Rolling-origin evaluation of the baselines

In [ ]:
from src.data.loader import rolling_origin_folds
for f in rolling_origin_folds(cfg, panel):
    print(f'{f.name}: origin {f.origin.date()} -> predict {f.start.date()}..{f.end.date()}')

In [ ]:
from src.evaluation.validate import walk_forward, aggregate_scores, per_fold_table
from src.evaluation.metrics import skill_vs
res = walk_forward(cfg, panel, fit_ml=False)
skill_vs(aggregate_scores(res), 'seasonal_naive').round(4)

In [ ]:
per_fold_table(res).round(4)

## Croston is not a moving average

A naive implementation collapses to one: mean non-zero size x non-zero rate equals sum/n, the window mean. Real Croston smooths size and interval separately and updates only when a sale occurs. SBA then corrects Croston's known upward bias.

In [ ]:
import numpy as np, pandas as pd
from src.features.build import croston_state
dates = pd.date_range('2020-01-01', periods=90, freq='D')
y = np.where(np.arange(90) % 5 == 0, 10.0, 0.0)
wide = pd.DataFrame([y], index=['S'], columns=dates)
print(croston_state(wide, alpha=0.1))
print('28-day moving average:', y[-28:].mean())

## Watch the `zero` baseline's MASE

Predicting zero everywhere attains the **best MASE** on this data, because averaging per-row scaled errors rewards predicting nothing on sparse series. WAPE correctly scores it 1.0. This is why WAPE is the headline metric here.